# 🎨 Tune-Text AI Studio Cloud Worker (Google Colab T4 GPU)
### Tự động tạo Hình ảnh (Text-to-Image 8K) & Video chuyển động (Image-to-Video MP4)
---
Sổ tay này biến **Google Colab miễn phí** thành máy chủ Cloud GPU riêng biệt cho Tune-Text.
- **Không tốn 1% CPU/RAM của máy tính bạn**
- **Không kiểm duyệt, không giới hạn số lượng ảnh/video**
- **Không cần mở Chrome phụ hay cào web bên ngoài**

👉 **Hướng dẫn 1 bước:** Bấm menu **Runtime** ➔ **Run all** (hoặc nhấn `Ctrl + F9`). Cuộn xuống cuối để lấy đường dẫn `https://xxx.trycloudflare.com` dán vào Tune-Text!

In [ ]:
# BƯỚC 1: KIỂM TRA GPU VÀ CÀI ĐẶT THƯ VIỆN
!nvidia-smi
!pip install -q diffusers transformers accelerate safetensors huggingface_hub fastapi uvicorn pydantic torchvision

In [ ]:
# BƯỚC 2: CÀI ĐẶT CLOUDFLARE TUNNEL (ĐỂ TẠO LINK TRUY CẬP MIỄN PHÍ TỪ XA)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Đã cài đặt Cloudflare Tunnel thành công!")

In [ ]:
# BƯỚC 3: KHỞI TẠO SERVER FASTAPI SINH ẢNH & VIDEO
server_code = '''
import os, io, time, base64, tempfile, torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from PIL import Image

app = FastAPI(title="Tune-Text AI Studio", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
pipe_image = None
pipe_video = None

def get_gpu_name():
    if torch.cuda.is_available():
        return f"{torch.cuda.get_device_name(0)} ({round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)} GB VRAM)"
    return "CPU"

def load_image_pipeline():
    global pipe_image
    if pipe_image is not None: return pipe_image
    print("⏳ Đang nạp mô hình Text-to-Image (SDXL Lightning)...\n")
    from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
    from huggingface_hub import hf_hub_download
    base = "stabilityai/stable-diffusion-xl-base-1.0"
    repo = "ByteDance/SDXL-Lightning"
    ckpt = "sdxl_lightning_4step_unet.safetensors"
    unet = UNet2DConditionModel.from_config(base, subfolder="unet").to(DEVICE, DTYPE)
    unet.load_state_dict(torch.load(hf_hub_download(repo, ckpt), map_location=DEVICE))
    pipe_image = StableDiffusionXLPipeline.from_pretrained(base, unet=unet, torch_dtype=DTYPE, variant="fp16").to(DEVICE)
    pipe_image.scheduler = EulerDiscreteScheduler.from_config(pipe_image.scheduler.config, timestep_spacing="trailing")
    if torch.cuda.is_available(): pipe_image.enable_attention_slicing()
    print("✅ Đã nạp thành công mô hình Text-to-Image!\n")
    return pipe_image

def load_video_pipeline():
    global pipe_video
    if pipe_video is not None: return pipe_video
    print("⏳ Đang nạp mô hình Image-to-Video (SVD)...\n")
    from diffusers import StableVideoDiffusionPipeline
    pipe_video = StableVideoDiffusionPipeline.from_pretrained("stabilityai/stable-video-diffusion-img2vid-xt", torch_dtype=DTYPE, variant="fp16").to(DEVICE)
    if torch.cuda.is_available(): pipe_video.enable_model_cpu_offload()
    print("✅ Đã nạp thành công mô hình Image-to-Video!\n")
    return pipe_video

class ImgReq(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "ugly, blurry, low quality, distorted"
    aspect_ratio: Optional[str] = "16:9"
    steps: Optional[int] = 4

class VidReq(BaseModel):
    image_base64: str
    prompt: Optional[str] = ""
    num_frames: Optional[int] = 25
    fps: Optional[int] = 8

@app.get("/health")
@app.get("/info")
def health():
    return {"status": "ready", "gpu": get_gpu_name(), "models": ["SDXL-Lightning", "SVD-Video"]}

@app.post("/api/generate-image")
def gen_image(req: ImgReq):
    pipe = load_image_pipeline()
    w, h = 1024, 576
    if req.aspect_ratio == "9:16": w, h = 576, 1024
    elif req.aspect_ratio == "1:1": w, h = 1024, 1024
    start = time.time()
    with torch.inference_mode():
        img = pipe(prompt=req.prompt, negative_prompt=req.negative_prompt, width=w, height=h, num_inference_steps=req.steps or 4, guidance_scale=0.0).images[0]
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return {"success": True, "image_base64": f"data:image/png;base64,{b64}", "elapsed_seconds": round(time.time() - start, 2)}

@app.post("/api/generate-video")
def gen_video(req: VidReq):
    pipe = load_video_pipeline()
    from diffusers.utils import export_to_video
    raw = req.image_base64.split(",", 1)[1] if "," in req.image_base64 else req.image_base64
    inp = Image.open(io.BytesIO(base64.b64decode(raw))).convert("RGB").resize((1024, 576))
    start = time.time()
    with torch.inference_mode():
        frames = pipe(inp, decode_chunk_size=8, motion_bucket_id=127, fps=req.fps or 8, num_frames=req.num_frames or 25).frames[0]
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tf: tmp_p = tf.name
    export_to_video(frames, tmp_p, fps=req.fps or 8)
    with open(tmp_p, "rb") as f: vid_b = f.read()
    try: os.remove(tmp_p)
    except: pass
    b64 = base64.b64encode(vid_b).decode("utf-8")
    return {"success": True, "video_base64": f"data:video/mp4;base64,{b64}", "elapsed_seconds": round(time.time() - start, 2)}
'''

with open("server.py", "w") as f:
    f.write(server_code)
print("✅ Đã tạo tệp server.py thành công!")

In [ ]:
# BƯỚC 4: KHỞI ĐỘNG SERVER VÀ MỞ ĐƯỜNG DẪN CLOUDFLARE PUBLIC
import subprocess, time, re

# Khởi động FastAPI ngầm ở port 8000
server_proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Khởi động Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

print("\n⏳ Đang tạo đường dẫn Cloudflare Tunnel...")
public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n===================================================================")
    print("🎉 CHÚC MỪNG BẠN! CLOUD GPU WORKER ĐÃ HOẠT ĐỘNG HOÀN TOÀN!")
    print(f"🔗 Đường dẫn kết nối Tune-Text: {public_url}")
    print("===================================================================")
    print("👉 Hãy sao chép đường dẫn trên và dán vào ô 'Cloudflare URL' trong Tune-Text Web!")
else:
    print("⚠️ Chưa lấy được URL tự động, bạn có thể kiểm tra trong log của cloudflared.")